# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR<sup>2</sup> dataset, described using a Croissant schema, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant -q

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset's Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print summary
print(f"Dataset title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print("License:", dataset.metadata.license)
print("DOI:", dataset.metadata.identifier)
print("Date Published:", dataset.metadata.datePublished)


## 2. Data Overview
Let's explore the dataset structure: record set `@id`s and fields in this Croissant package.

We enumerate all available record sets and their fields, referencing entities by their `@id` as per schema best practices.

In [ ]:
# List available record sets and their fields/columns
from pprint import pprint

# The Croissant schema supports `record_sets` as a list of objects with @id and field/column info
record_sets = dataset.metadata.recordSets
if not record_sets:
    print("No record sets defined in the dataset metadata.")
else:
    for rs in record_sets:
        print(f'RecordSet @id: {rs["@id"]}')
        print(f'  Name: {rs.get("name", "-")}' )
        # List fields and columns
        if hasattr(rs, 'fields') or 'fields' in rs:
            fields = rs.fields if hasattr(rs, 'fields') else rs['fields']
            print("  Fields/Columns:")
            for field in fields:
                field_id = field['@id'] if isinstance(field, dict) else field.@id
                print(f'    - Field @id: {field_id}')
        print("")

# If no record sets found, print the list of objects in the dataset
if not record_sets:
    print("Available attributes in dataset.metadata:")
    pprint(vars(dataset.metadata).keys())

## 2b. Explore a Sample Record
You can preview the first record from the main record set. For this, you'll need the `@id` of a record set. In this dataset, many Croissant datasets use the main table as the only record set.

Let's enumerate to find a record set to preview.

In [ ]:
# Find the main record set's @id (if any)
if dataset.metadata.recordSets:
    main_recordset_id = dataset.metadata.recordSets[0]["@id"]
    print(f"Using main record set with @id: {main_recordset_id}")
    sample_record = next(dataset.records(record_set=main_recordset_id))
    pprint(sample_record)
else:
    print("No record sets available for preview.")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. We use the appropriate `@id`s for record set and fields.

If the dataset contains multiple record sets, you can load each one into a DataFrame. Here, we illustrate for the main record set.

In [ ]:
# List all record set IDs
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSets] if dataset.metadata.recordSets else []
print("RecordSet @id's:", record_set_ids)

# Load records for each record set into a DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} rows.")
    dataframes[record_set_id] = df

# Show columns in the first record set
if record_set_ids:
    first_id = record_set_ids[0]
    print(f"Columns for {first_id}:\n", dataframes[first_id].columns.tolist())
    display(dataframes[first_id].head())
else:
    print("No records to load.")

## 4. Exploratory Data Analysis (EDA)
In this section, we demonstrate filtering, normalization, and grouping operations for quantitative exploration. Use the proper field `@id`s when selecting and transforming data. 

Below, choose a numeric field and a group-by field from the displayed DataFrame columns (referenced by `@id`).

In [ ]:
# Choose a record set
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"Running EDA for main record set: {record_set_id}")
    print("Columns:", df.columns.tolist())

    # Select a numeric field by @id (example: 'cr:Peptides') and a group field ('cr:Description' or similar)
    # Below, try common protein dataset fields, adapt as needed for your schema:
    possible_numeric_fields = [col for col in df.columns if 'coverage' in col.lower() or 'abundance' in col.lower() or 'peptide' in col.lower() or 'mw' in col.lower() or 'intensity' in col.lower()]
    if not possible_numeric_fields:
        possible_numeric_fields = df.select_dtypes(include=['number']).columns.tolist()

    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
        print(f"Using field '{numeric_field}' for filtering and normalization.")
    else:
        print("No obvious numeric fields detected.")
        numeric_field = df.columns[0]

    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    filtered_df = df[df[numeric_field]>threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    import numpy as np
    if pd.api.types.is_numeric_dtype(df[numeric_field]):
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean())/filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a textual or categorical field (e.g., 'cr:Description' or similar)
    group_fields = [c for c in df.columns if 'sample' in c.lower() or 'description' in c.lower() or 'group' in c.lower()]
    if group_fields:
        group_field = group_fields[0]
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        display(grouped_df.head())
    else:
        print("No clear grouping field detected.")
else:
    print("No data available for EDA.")

## 5. Visualization
Let's plot the distribution of the chosen numeric field and compare it between groups if a group field is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group if group_field available
    if group_fields:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric data to plot.")

## 6. Conclusion
- We have loaded the FAIR² Croissant dataset and inspected its metadata structure.
- Record sets and their corresponding fields (referenced by `@id`) were enumerated.
- Tabular records were extracted into Pandas DataFrames, processed, filtered, normalized, and grouped using standard data science practices.
- Simple visualizations enabled quick exploration of numeric distributions and group differences.

> **Tip:** Always reference entities by `@id` as you work with Croissant datasets in `mlcroissant` workflows, allowing traceability and reproducibility for FAIR data principles.
